# M2 Generators — Qualitative Side-by-Side Comparison

Pick N pairs from the eval set. For each, show:
- the thumbnail image
- 3-4 frames sampled from the video
- the ground-truth QVH query
- queries produced by each M2 you're comparing
- the resulting consistency score from each M2

Use this for: paper figures, eyeballing failure modes, M2 generation
inspection.


In [ ]:
# [COLAB_BOOTSTRAP_v2_modular]
# Each Colab session:
#   1. Mount Drive             (Drive has the repo + the tar.gz of M1 features)
#   2. cd into the repo        (so relative paths to src/, configs/, etc. work)
#   3. Symlink Drive data dirs (processed/, qvh_frames/, etc.) into the repo
#   4. EXTRACT M1 features tar.gz from Drive ->  LOCAL SSD  (/content/features_local)
#      Reason: 19,000 small .npz files corrupt on Drive's FUSE; LOCAL SSD is fast.
#   5. Clone upstream moment_detr code (for build_model imports)
#
# Drive is ONLY used as the SOURCE of the tar.gz. Features ALWAYS live on
# /content/ (LOCAL SSD) after extraction. Same pattern as
# run_pipeline_allinone.ipynb cell 2.5 and sanity_consistency.ipynb cell 0.

import os, sys, subprocess, time, tarfile, shutil
from pathlib import Path

# ----- 1. Mount Drive ----------------------------------------------------
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

# ----- 2. Find repo + cd ------------------------------------------------
DRIVE_ROOT = Path('/content/drive/MyDrive/thumbnail_project')
REPO_DIR   = DRIVE_ROOT / 'thumbnail-consistency'
assert REPO_DIR.exists(), f"Upload your repo folder to {REPO_DIR} first."

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# ----- 3. Symlink data dirs ---------------------------------------------
for d in ['processed', 'outputs', 'qvh_videos', 'qvh_frames', 'qvh_thumbnails',
          'qvh_frames_within_neg', 'qvh_frames_multi', 'checkpoints']:
    src, dst = DRIVE_ROOT / d, REPO_DIR / d
    if src.exists() and not dst.exists() and not dst.is_symlink():
        try: os.symlink(src, dst)
        except OSError: pass

# ----- 4. Extract M1 features from Drive .tar.gz -> LOCAL SSD ----------
# Source on Drive (the 7.5 GB archive):
ARCHIVE_ON_DRIVE = DRIVE_ROOT / 'moment_detr_features.tar.gz'
# Destination on LOCAL SSD (NOT on Drive — Drive can't hold 19k small files):
EXTRACTED_LOCAL  = Path('/content/features_local')

def _find_feature_dirs(root):
    """Look inside `root` for clip_features/ and clip_text_features/ dirs
    that contain .npz files. Returns (clip_dir, text_dir) paths or (None, None)."""
    clip = text = None
    for r, dirs, _ in os.walk(root):
        for d in dirs:
            p = Path(r) / d
            if d == 'clip_features' and len(list(p.glob('*.npz'))) > 1000:
                clip = p
            if d == 'clip_text_features' and len(list(p.glob('*.npz'))) > 1000:
                text = p
    return clip, text

# Check ONLY local SSD (NOT Drive) for an already-extracted copy from earlier
# in THIS Colab session. If you restart the runtime, /content is wiped and
# this check returns (None, None), triggering a fresh extraction.
CLIP_FEAT_DIR, TEXT_FEAT_DIR = (None, None)
if EXTRACTED_LOCAL.exists():
    CLIP_FEAT_DIR, TEXT_FEAT_DIR = _find_feature_dirs(EXTRACTED_LOCAL)

if CLIP_FEAT_DIR and TEXT_FEAT_DIR:
    print(f"[features] Found local extraction from earlier this session (skip).")
    print(f"           {EXTRACTED_LOCAL}")
else:
    assert ARCHIVE_ON_DRIVE.exists(), (
        f"Source archive missing on Drive: {ARCHIVE_ON_DRIVE}\n"
        f"This is the only file the bootstrap reads from Drive."
    )
    # Wipe any stale partial extraction and start fresh
    if EXTRACTED_LOCAL.exists():
        shutil.rmtree(EXTRACTED_LOCAL)
    EXTRACTED_LOCAL.mkdir(parents=True)
    sz_gb = ARCHIVE_ON_DRIVE.stat().st_size / 1e9
    print(f"[features] Extracting {sz_gb:.2f} GB:")
    print(f"           FROM (Drive)     : {ARCHIVE_ON_DRIVE}")
    print(f"           TO   (local SSD) : {EXTRACTED_LOCAL}")
    t0 = time.time()
    with tarfile.open(ARCHIVE_ON_DRIVE) as tf:
        tf.extractall(EXTRACTED_LOCAL)
    print(f"[features] Done in {time.time()-t0:.0f}s.")
    CLIP_FEAT_DIR, TEXT_FEAT_DIR = _find_feature_dirs(EXTRACTED_LOCAL)

assert CLIP_FEAT_DIR and TEXT_FEAT_DIR, (
    f"Feature dirs not found after extraction. Check {EXTRACTED_LOCAL}"
)

# ----- 5. Clone upstream Moment-DETR (for build_model imports) ---------
MD_DIR = Path('/content/moment_detr_upstream')
if not MD_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jayleicn/moment_detr.git',
                    str(MD_DIR)], check=True)
if str(MD_DIR) not in sys.path:
    sys.path.insert(0, str(MD_DIR))

# ----- 6. Verification summary -----------------------------------------
import torch
print()
print(f"Repo            : {REPO_DIR}")
print(f"Device          : {'cuda' if torch.cuda.is_available() else 'cpu (WARN: slow)'}")
print(f"Drive tar.gz    : {ARCHIVE_ON_DRIVE}  ({sz_gb:.2f} GB)" if 'sz_gb' in dir() else "")
print(f"clip_features   : {CLIP_FEAT_DIR}  ({len(list(CLIP_FEAT_DIR.glob('*.npz'))):,} files)")
print(f"clip_text_feats : {TEXT_FEAT_DIR}  ({len(list(TEXT_FEAT_DIR.glob('*.npz'))):,} files)")
print(f"src/ importable : {(REPO_DIR / 'src' / '__init__.py').exists()}")


## 1. Install M2 dependencies (only if you'll use a VLM)


In [ ]:
import sys, subprocess
USE_VLM = True   # ★ set False to skip if comparing only lightweight M2s
if USE_VLM:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'transformers>=4.45', 'accelerate>=0.30', 'bitsandbytes>=0.43',
                    'qwen-vl-utils', 'sentencepiece'], check=True)
    print('VLM deps installed.')
else:
    print('Skipped VLM deps.')


## 2. Choose M2s to compare and an eval set

In [ ]:
M2_LIST   = ['gt', 'random']     # add 'clipcap' / 'qwen2vl_simple' etc.
EVAL_PATH = 'processed/consistency_eval.jsonl'
N_SAMPLES = 5

M2_KWARGS = {
    'gt'             : {'lookup_jsonl': EVAL_PATH},
    'random'         : {'pool_jsonl'  : EVAL_PATH, 'seed': 2026},
    'clipcap'        : {'ckpt_path'   : 'checkpoints/clipcap_best.pt'},
    'qwen2vl_simple' : {'model_id': 'Qwen/Qwen2-VL-7B-Instruct',          'load_in_4bit': True},
    'qwen2vl_aspect' : {'model_id': 'Qwen/Qwen2-VL-7B-Instruct',          'load_in_4bit': True},
    'llava16_aspect' : {'model_id': 'llava-hf/llava-v1.6-mistral-7b-hf',  'load_in_4bit': True},
}
print(f'Comparing {M2_LIST} on {N_SAMPLES} pairs from {EVAL_PATH}')


## 2. Build scorer once + all M2s up front

In [ ]:
import yaml, gc, torch
from src.m2_generators import build_m2
from src.models.moment_detr import MomentDETRFeaturePredictor
from src.scoring.consistency import ConsistencyScorer, ScoringConfig

cfg_yaml = yaml.safe_load(open('configs/consistency.yaml'))
m1 = MomentDETRFeaturePredictor(
    ckpt_path=cfg_yaml['paths']['m1_ckpt'], repo_root=str(REPO_DIR),
)
scoring_cfg = ScoringConfig(alpha=cfg_yaml['alpha'], tau=cfg_yaml['tau'], top_p=cfg_yaml['top_p'])
scorer = ConsistencyScorer(m1, str(CLIP_FEAT_DIR), scoring_cfg)

# NOTE: heavy M2s (Qwen, LLaVA) loaded together can OOM. If so:
#   1. comment out the heavy ones in M2_LIST
#   2. restart runtime
#   3. load one at a time
m2_instances = {}
for name in M2_LIST:
    print(f'  building {name}...')
    m2_instances[name] = build_m2(name, **M2_KWARGS.get(name, {}))
print('Done.')


## 3. Sample pairs and run all M2s on them

In [ ]:
import json, random, time
random.seed(7)

pairs = [json.loads(l) for l in open(EVAL_PATH, encoding='utf-8') if l.strip()]
positives = [p for p in pairs if p['label'] == 1]
sampled = random.sample(positives, min(N_SAMPLES, len(positives)))
print(f'Sampled {len(sampled)} positive pairs')

table = []   # one row per pair, with all M2 outputs
for p in sampled:
    row = {
        'pair_id'   : p['pair_id'],
        'thumb_vid' : p['thumb_vid'],
        'video_vid' : p['video_vid'],
        'frame_path': p['frame_path'],
        'gt_query'  : p['thumb_query'],
        'm2_runs'   : {},
    }
    for name, m2 in m2_instances.items():
        gen_kwargs = {'qid': p['thumb_qid']} if name == 'gt' else {}
        t0 = time.time()
        qs = m2.generate(p['frame_path'], k=3, **gen_kwargs)
        gen_ms = (time.time() - t0) * 1000
        out = scorer.score(thumbnail_path=p['frame_path'],
                            vid=p['video_vid'], queries=qs)
        row['m2_runs'][name] = {
            'queries': qs, 'score': float(out['C']), 'gen_ms': gen_ms,
        }
    table.append(row)
print('Scored all pairs.')


## 4. Render side-by-side panels

In [ ]:
import matplotlib.pyplot as plt
import cv2, textwrap
from PIL import Image

VIDEOS_DIR = REPO_DIR / 'qvh_videos'

def sample_frames(video_path, n=4):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened(): return []
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
    out = []
    for i in range(n):
        target = int(total * (i + 0.5) / n) if total else 0
        cap.set(cv2.CAP_PROP_POS_FRAMES, target)
        ret, fr = cap.read()
        if ret:
            out.append(Image.fromarray(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)))
    cap.release()
    return out


for row in table:
    n_vid_frames = 4
    n_cols = 1 + n_vid_frames
    fig, axes = plt.subplots(1, n_cols, figsize=(3.3 * n_cols, 3.3))

    # Thumb
    try:
        axes[0].imshow(Image.open(row['frame_path']).convert('RGB'))
        axes[0].set_title(f"THUMB\n{row['thumb_vid'][:22]}", fontsize=9)
    except Exception:
        axes[0].text(0.5, 0.5, 'thumb missing', ha='center')
    axes[0].axis('off')

    # Video frames
    vp = VIDEOS_DIR / f"{row['video_vid']}.mp4"
    vf = sample_frames(vp, n_vid_frames) if vp.exists() else []
    for j in range(n_vid_frames):
        ax = axes[1 + j]
        if j < len(vf):
            ax.imshow(vf[j])
            ax.set_title(f'VIDEO @ ~{int(100*(j+0.5)/n_vid_frames)}%', fontsize=8)
        else:
            ax.text(0.5, 0.5, 'N/A', ha='center')
        ax.axis('off')

    # Title with all M2 outputs
    lines = [f"pair_id={row['pair_id']}    GT: {row['gt_query']}"]
    for name, info in row['m2_runs'].items():
        lines.append(f"\n[{name}]  score={info['score']:+.3f}  ({info['gen_ms']:.0f} ms)")
        for i, q in enumerate(info['queries']):
            lines.append(f"    q{i}: {textwrap.shorten(q, 90)}")
    fig.suptitle("\n".join(lines), fontsize=8.5, y=1.0 + 0.08*len(row['m2_runs']))
    plt.tight_layout()
    plt.show()


## 5. Optional — save the comparison as JSON for the report

In [ ]:
out_path = REPO_DIR / 'outputs' / 'predictions' / 'qual_comparison.jsonl'
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, 'w', encoding='utf-8') as f:
    for row in table:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')
print(f'Saved {out_path}')
